In [ ]:
import os
import sys
import json
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm

## Analyze SNR

In [ ]:
for subset_name in ['tst']:
    subset_time = 0
    subset_path = os.path.join(r"/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference", subset_name)
    list_snrs = []
    for sample_name in tqdm(os.listdir(subset_path)):
        overall_noise_path = os.path.join(subset_path, sample_name, 'ambient_noise.flac')
        target_speech_path = os.path.join(subset_path, sample_name, 'target_speech.flac')
        overall_noise, _ = librosa.load(overall_noise_path, sr=16000, mono=False)
        target_speech, _ = librosa.load(target_speech_path, sr=16000, mono=False)
        overall_noise_power = overall_noise.var()
        target_speech_power = target_speech.var()
        snr = 10 * np.log10(target_speech_power / overall_noise_power)
        list_snrs.append(snr)

In [ ]:
# Plot histogram of SNRs
plt.figure(figsize=(10, 6))
plt.hist(list_snrs, bins=20, edgecolor='black', range=(-10, 10))
plt.title('Distribution of SNRs in Test Set')
plt.xlabel('SNR [dB]')
plt.ylabel('Number of Samples')
plt.grid(axis='y', alpha=0.75)
plt.show()

## Analyze Duration

In [ ]:
for subset_name in ['tst']:
    list_durations = []
    subset_time = 0
    subset_path = os.path.join(r"/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference", subset_name)
    for sample_name in os.listdir(subset_path):
        sample_path = os.path.join(subset_path, sample_name, 'noisy_mixture.flac')
        sample_time = librosa.get_duration(path=sample_path, sr=16000)
        list_durations.append(sample_time)
        subset_time += sample_time
    print(f"{subset_name}: {subset_time:.2f} seconds")

In [ ]:
# Plot histogram of durations.
plt.figure(figsize=(10, 6))
plt.hist(list_durations, bins=20, edgecolor='black', range=(8, 20))
plt.title('Distribution of Sample Durations in Test Set')
plt.xlabel('Time [s]')
plt.ylabel('Number of Samples')
plt.grid(axis='y', alpha=0.75)
plt.show()

## Adjust Metadata

In [ ]:
for subset_name in ['tst', 'trn', 'val']:
    subset_path = os.path.join(r"/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference", subset_name)
    noise_metadata_path = os.path.join(r"/home/ovistetom/Documents/data/WHAM/original/metadata", f"noise_meta_{subset_name}.csv")
    noise_metadata = pd.read_csv(noise_metadata_path)
    for sample_name in tqdm(os.listdir(subset_path)):
        sample_path = os.path.join(subset_path, sample_name)
        with open(os.path.join(sample_path, 'metadata.json'), 'r') as f:
            sample_metadata = json.load(f)
        loc_id = noise_metadata[noise_metadata['utterance_id'] == (sample_metadata['NOISE_ID'].split('sp')[0]+ '.wav')]['Location ID'].item()
        sample_metadata['LOC_ID'] = loc_id
        with open(os.path.join(sample_path, 'metadata.json'), 'w') as f:
            json.dump(sample_metadata, f, indent=4)
        try:
            os.remove(os.path.join(sample_path, 'metadata2.json'))
        except FileNotFoundError:
            pass

In [ ]:
for subset_name in ['trn']:
    subset_path = os.path.join(r"/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference", subset_name)
    noise_metadata_path = os.path.join(r"/home/ovistetom/Documents/data/WHAM/original/metadata", f"noise_meta_{subset_name}.csv")
    noise_metadata = pd.read_csv(noise_metadata_path)

In [ ]:
loc_id = 0
for index, row in noise_metadata.iterrows():
    # if row['utterance_id'].startswith(sample_metadata['NOISE_ID']):
    #     loc_id = row['Location ID']
    if row['utterance_id'] == "01la0107_0.30584_02co0307_-0.30584.wav":
        loc_id = row['Location ID']
        break
print(loc_id)

## Re-Generate (Adjust SNR and SIR)

In [ ]:
def adjust_metadata(
        path_src: str,
        path_dst: str,
        new_sir: float,
        new_snr: float,
):
    with open(os.path.join(path_src, 'metadata.json'), 'r') as f:
          = json.load(f)
    with open(os.path.join(path_dst, 'metadata.json'), 'w') as f:
        metadata['SIR_DB'] = round(new_sir)
        metadata['SNR_DB'] = round(new_snr)
        json.dump(metadata, f, indent=4)
    

In [ ]:
for subset_name in ['tst', 'trn', 'val']:
    subset_time = 0
    subset_path_src = os.path.join(r"/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference", subset_name)
    subset_path_dst = os.path.join(r"/home/ovistetom/Documents/data/MIX-EARS-WHAM/reference_0_6", subset_name)
    list_snrs = []
    for sample_name in tqdm(os.listdir(subset_path_src)):
        sample_path_src = os.path.join(subset_path_src, sample_name)
        target_speech_path = os.path.join(sample_path_src, 'target_speech.flac')
        reverb_speech_path = os.path.join(sample_path_src, 'reverb_speech.flac')
        interf_speech_path = os.path.join(sample_path_src, 'interf_speech.flac')
        ambient_noise_path = os.path.join(sample_path_src, 'ambient_noise.flac')

        # Load signals.
        x, _ = librosa.load(target_speech_path, sr=16000, mono=False)
        r, _ = librosa.load(reverb_speech_path, sr=16000, mono=False)
        d, _ = librosa.load(interf_speech_path, sr=16000, mono=False)
        v, _ = librosa.load(ambient_noise_path, sr=16000, mono=False)

        # Adjust SNR and SIR.
        snr = np.power(x, 2).sum() / np.power(v, 2).sum()
        sir = np.power(x, 2).sum() / np.power(d, 2).sum()
        target_snr = 10**(0.1*np.random.uniform(0, 6))
        target_sir = 10**(0.1*np.random.uniform(0, 6))
        v = v * np.sqrt(snr / target_snr)
        d = d * np.sqrt(sir / target_sir)
        n = r + v + d
        y = x + n

        # Save new signals.
        max_amplitude = 2*np.abs(y).max()
        sample_path_dst = os.path.join(subset_path_dst, sample_name)
        os.makedirs(sample_path_dst, exist_ok=True)
        target_speech_path_dst = os.path.join(sample_path_dst, 'target_speech.flac')
        reverb_speech_path_dst = os.path.join(sample_path_dst, 'reverb_speech.flac')
        interf_speech_path_dst = os.path.join(sample_path_dst, 'interf_speech.flac')
        ambient_noise_path_dst = os.path.join(sample_path_dst, 'ambient_noise.flac')
        noisy_mixture_path_dst = os.path.join(sample_path_dst, 'noisy_mixture.flac')
        overall_noise_path_dst = os.path.join(sample_path_dst, 'overall_noise.flac')

        adjust_metadata(path_src=sample_path_src, path_dst=sample_path_dst, new_sir=target_sir, new_snr=target_snr)

        for data, path in zip([x, r, d, v, n, y], [target_speech_path_dst, reverb_speech_path_dst, interf_speech_path_dst, ambient_noise_path_dst, overall_noise_path_dst, noisy_mixture_path_dst]):
            sf.write(path, (data / max_amplitude).T, samplerate=16000)
        

